# Install dependencies from uv and setup reloading of imports.

In [1]:
!uv sync
%load_ext autoreload
%autoreload 2

Resolved 133 packages in 8ms
Checked 130 packages in 208ms


# Test route tool

In [ ]:
import tools

print("================ NYC to Trenton NJ =======================")
tools.get_routes("New York City, NY", "Trenton, NJ")

================ NYC to Trenton NJ =======================


'{\n  "origin": "New York City, NY",\n  "destination": "Trenton, NJ",\n  "travel_mode": "DRIVE",\n  "routes": [\n    {\n      "route_number": 1,\n      "name": "Route 1",\n      "description": "I-95 S",\n      "duration": "1h 17m",\n      "distance_km": 108.2,\n      "warnings": [\n        "This route has tolls.",\n        "This route includes a highway."\n      ],\n      "steps": [\n        {\n          "instruction": "Head southeast toward Park Row\\nPartial restricted usage road",\n          "distance_km": 0.06,\n          "duration_minutes": 0.2\n        },\n        {\n          "instruction": "Turn right onto Park Row",\n          "distance_km": 0.23,\n          "duration_minutes": 1.4\n        },\n        {\n          "instruction": "Continue straight onto Barclay St",\n          "distance_km": 0.15,\n          "duration_minutes": 0.9\n        },\n        {\n          "instruction": "Turn right onto Church St",\n          "distance_km": 0.35,\n          "duration_minutes": 1.6\n 

# Create the agents.

In [80]:
from google.adk import Agent, Workflow
from google.adk.tools import google_search, agent_tool

import config
import instructions
import tools
import callbacks
import workflow_nodes

input_guard_agent = Agent(
    name="input_guard_agent",
    model=config.GEMINI_LITE_MODEL,
    instruction=instructions.get_agent_instructions("fema-input-guard"),
    before_model_callback=callbacks.logging_before_callback,
    after_model_callback=callbacks.logging_after_callback,
    output_schema=workflow_nodes.GuardOutput)

weather_agent = Agent(
    name="weather_agent",
    model=config.GEMINI_MODEL,
    instruction=instructions.get_agent_instructions("weather-agent-instructions"),
    before_model_callback=callbacks.logging_before_callback,
    after_model_callback=callbacks.logging_after_callback,
    tools=[tools.get_weather, tools.get_lat_lon]
)

route_agent = Agent(
    name="route_agent",
    model=config.GEMINI_MODEL,
    instruction=instructions.get_agent_instructions("route-agent"),
    before_model_callback=callbacks.logging_before_callback,
    after_model_callback=callbacks.logging_after_callback,
    tools=[tools.get_routes]
)

search_agent_tool = Agent(name="search_agent_tool",
    model=config.GEMINI_MODEL,
    instruction=instructions.get_agent_instructions("research-agent"),
    before_model_callback=callbacks.logging_before_callback,
    after_model_callback=callbacks.logging_after_callback,
    tools=[google_search]
)

search_agent = Agent(
    name="search_agent",
    model=config.GEMINI_MODEL,
    instruction=instructions.get_agent_instructions("research-agent"),
    before_model_callback=callbacks.logging_before_callback,
    after_model_callback=callbacks.logging_after_callback,
    tools=[agent_tool.AgentTool(agent=search_agent_tool)])

general_questions_agent = Agent(
    name="general_questions_agent",
    model=config.GEMINI_MODEL,
    instruction=instructions.get_agent_instructions("fema-general-questions"),
    before_model_callback=callbacks.logging_before_callback,
    after_model_callback=callbacks.logging_after_callback)

refiner_agent = Agent(
    name="refiner_agent",
    model=config.GEMINI_MODEL,
    instruction=instructions.get_agent_instructions("fema-refiner"),
    before_model_callback=callbacks.logging_before_callback,
    after_model_callback=callbacks.logging_after_callback
)

router_agent = Agent(
    name="router_agent",
    model=config.GEMINI_MODEL,
    instruction=instructions.get_agent_instructions("fema-router-agent"),
    before_model_callback=callbacks.logging_before_callback,
    after_model_callback=callbacks.logging_after_callback,
    sub_agents=[
        weather_agent, route_agent, search_agent, general_questions_agent
    ])

root_agent = Workflow(name="root_agent",
    edges = [
        ("START", input_guard_agent, workflow_nodes.fema_guard_node, router_agent, refiner_agent)
    ]
)


# Perform tests on local agent.

In [81]:
import agent_tester

tester = agent_tester.AgentTester(root_agent)

await tester.run_prompt("Search online to see if there currently any emergencies in the Reston VA area? What is the best way to get from there to Washington D.C?")

logging_before_callback- Agent: input_guard_agent, User entered: Search online to see if there currently any emergencies in the Reston VA area? What is the best way to get from there to Washington D.C?
logging_after_callback- Model response: {
  "allowed": true,
  "text": "Search online to see if there currently any emergencies in the Reston VA area? What is the best way to get from there to Washington D.C?"
}


{ "allowed": true, "text": "Search online to see if there currently any emergencies in the Reston VA area? What is 
the best way to get from there to Washington D.C?" }

{'allowed': True, 'text': 'Search online to see if there currently any emergencies in the Reston VA area? What is the best way to get from there to Washington D.C?'}


logging_before_callback- Agent: router_agent, User entered: Search online to see if there currently any emergencies in the Reston VA area? What is the best way to get from there to Washington D.C?


logging_before_callback- Agent: route_agent, User entered: For context:


...

logging_before_callback- Agent: route_agent, User entered: For context:


logging_after_callback- Model response: 🗺️ Routes from Reston, VA to Washington D.C.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
🥇 Route 1 — Recommended (Shortest)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📏 Distance:        22.37 miles
⏱️ Est. Travel Time: 0 hours 33 minutes
🛣️ Via:             VA-267 E and I-66 E
📝 Summary:         This route takes VA-267 E and I-66 E. It includes tolls and highways.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
🥈 Route 2
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📏 Distance:        23.49 miles
⏱️ Est. Travel Time: 0 hours 40 minutes
🛣️ Via:             I-66 E
📝 Summary:         This route takes I-66 E. It includes tolls and highways.


🗺️ Routes from Reston, VA to Washington D.C.                                                                       

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 🥇 Route 1 — Recommended (Shortest) ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 📏 Distance:   
22.37 miles ⏱️ Est. Travel Time: 0 hours 33 minutes 🛣️ Via:             VA-267 E and I-66 E 📝 Summary:            
This route takes VA-267 E and I-66 E. It includes tolls and highways.                                              

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 🥈 Route 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 📏 Distance:        23.49 miles ⏱️ Est. 
Travel Time: 0 hours 40 minutes 🛣️ Via:             I-66 E 📝 Summary:         This route takes I-66 E. It includes
tolls and highways.

logging_before_callback- Agent: refiner_agent, User entered: For context:
logging_after_callback- Model response: Here are the routes from Reston, VA to Washington D.C.:

*   **Recommended Route (Shortest):**
    *   **Distance:** 22.37 miles
    *   **Estimated Travel Time:** 33 minutes
    *   **Via:** VA-267 E and I-66 E
    *   **Details:** This route uses highways and includes tolls.

*   **Alternative Route:**
    *   **Distance:** 23.49 miles
    *   **Estimated Travel Time:** 40 minutes
    *   **Via:** I-66 E
    *   **Details:** This route also uses highways and includes tolls.


Here are the routes from Reston, VA to Washington D.C.:                                                            

 • Recommended Route (Shortest):                                                                                   
    • Distance: 22.37 miles                                                                                        
    • Estimated Travel Time: 33 minutes                                                                            
    • Via: VA-267 E and I-66 E                                                                                     
    • Details: This route uses highways and includes tolls.                                                        
 • Alternative Route:                                                                                              
    • Distance: 23.49 miles                                                                                        
    • Estimated Travel Time: 40 minutes                                                                            
    • Via: I-66 E                                                                                                  
    • Details: This route also uses highways and includes tolls.

In [82]:
import agent_tester

tester = agent_tester.AgentTester(root_agent)

await tester.run_prompt("What should I do if a forest file breaks out in my neighborhood? What is the best location to go to from Reston VA to get away from it?")

logging_before_callback- Agent: input_guard_agent, User entered: What should I do if a forest file breaks out in my neighborhood? What is the best location to go to from Reston VA to get away from it?
logging_after_callback- Model response: {
  "allowed": true,
  "text": "What should I do if a forest file breaks out in my neighborhood? What is the best location to go to from Reston VA to get away from it?"
}


{ "allowed": true, "text": "What should I do if a forest file breaks out in my neighborhood? What is the best      
location to go to from Reston VA to get away from it?" }

{'allowed': True, 'text': 'What should I do if a forest file breaks out in my neighborhood? What is the best location to go to from Reston VA to get away from it?'}


logging_before_callback- Agent: router_agent, User entered: What should I do if a forest file breaks out in my neighborhood? What is the best location to go to from Reston VA to get away from it?


logging_before_callback- Agent: general_questions_agent, User entered: For context:
logging_after_callback- Model response: If a wildfire breaks out in your neighborhood, your immediate priority is to ensure your safety and follow all instructions from local emergency services. Evacuate immediately if advised to do so, and proceed to designated safe areas.

**Key Actions:**

*   **Stay Informed:** Monitor local news, emergency alerts (like the Emergency Alert System - EAS, NOAA Weather Radio, or Wireless Emergency Alerts - WEA), and official social media accounts for updates and evacuation orders.
*   **Evacuate Immediately:** If an evacuation order is issued for your area, leave without delay. Do not wait for conditions to worsen.
*   **Follow Evacuation Routes:** Use the routes designated by emergency officials, as these are planned to be the safest. Avoid shortcuts that may lead you into danger.
*   **Prepare Your Home (if time allows and it's safe):**
    *   Close all windows and 

If a wildfire breaks out in your neighborhood, your immediate priority is to ensure your safety and follow all     
instructions from local emergency services. Evacuate immediately if advised to do so, and proceed to designated    
safe areas.                                                                                                        

Key Actions:                                                                                                       

 • Stay Informed: Monitor local news, emergency alerts (like the Emergency Alert System - EAS, NOAA Weather Radio, 
   or Wireless Emergency Alerts - WEA), and official social media accounts for updates and evacuation orders.      
 • Evacuate Immediately: If an evacuation order is issued for your area, leave without delay. Do not wait for      
   conditions to worsen.                                                                                           
 • Follow Evacuation Routes: Use the routes designated by emergency officials, as these are planned to be the      
   safest. Avoid shortcuts that may lead you into danger.                                                          
 • Prepare Your Home (if time allows and it's safe):                                                               
    • Close all windows and doors.                                                                                 
    • Turn off gas and propane.                                                                                    
    • Move combustible materials away from the house.                                                              
    • Turn on lights to make your home more visible to firefighters.                                               
 • Have an Emergency Kit Ready: Your "go-bag" should contain essential documents, medications, a first-aid kit,    
   food, water, a battery-powered radio, and chargers.                                                             
 • Communicate: Inform family and friends of your evacuation plans and destination.                                
 • Animal Safety: Plan for pets and livestock; if you evacuate, take them with you or ensure they are safely       
   secured.                                                                                                        

Important Details:                                                                                                 

 • Designated Safe Locations: The "best location to go to" from Reston, VA, or any area affected by wildfire, will 
   be determined by emergency management officials based on the fire's trajectory, wind direction, and availability
   of safe shelters. These locations are dynamic and will be communicated through official channels.               
 • Avoid Smoke: If you are not evacuating, stay indoors, keep windows and doors closed, and use an air purifier if 
   available to reduce smoke exposure.                                                                             
 • Community Preparedness: Wildfires can spread rapidly. Having a family emergency plan, including meeting points  
   and communication strategies, is crucial.                                                                       

Official Resources:                                                                                                

 • Ready.gov/wildfires: Provides comprehensive guidance on preparing for, surviving, and recovering from wildfires.
 • Local Emergency Management Agency: Check your local government's website for specific emergency plans and alerts
   for Reston, VA, and Fairfax County.                                                                             
 • FEMA: Visit DisasterAssistance.gov for information on federal assistance programs after a disaster.             

Do you need more specific guidance on preparing an emergency kit, creating a family evacuation plan, or            
understanding local alert systems?

logging_after_callback- Model response: I have addressed your previous question regarding what to do if a forest fire breaks out in your neighborhood and how to evacuate from Reston, VA.

**Summary of guidance provided:**

*   **Immediate Action:** Prioritize safety and follow emergency service instructions.
*   **Key Steps:** Stay informed via local alerts, evacuate immediately if ordered, follow designated routes, prepare your home (if safe), use your emergency kit, communicate with family, and consider pet safety.
*   **Evacuation Locations:** These are dynamic and will be communicated by emergency officials. Do not rely on pre-determined, static locations without official guidance.
*   **Resources:** Ready.gov/wildfires and your local emergency management agency are key resources.

If you have any further questions about specific aspects of wildfire preparedness, emergency kits, evacuation planning, or any other emergency-related topic, please let me know!


I have addressed your previous question regarding what to do if a forest fire breaks out in your neighborhood and  
how to evacuate from Reston, VA.                                                                                   

Summary of guidance provided:                                                                                      

 • Immediate Action: Prioritize safety and follow emergency service instructions.                                  
 • Key Steps: Stay informed via local alerts, evacuate immediately if ordered, follow designated routes, prepare   
   your home (if safe), use your emergency kit, communicate with family, and consider pet safety.                  
 • Evacuation Locations: These are dynamic and will be communicated by emergency officials. Do not rely on         
   pre-determined, static locations without official guidance.                                                     
 • Resources: Ready.gov/wildfires and your local emergency management agency are key resources.                    

If you have any further questions about specific aspects of wildfire preparedness, emergency kits, evacuation      
planning, or any other emergency-related topic, please let me know!

logging_before_callback- Agent: refiner_agent, User entered: For context:
logging_after_callback- Model response: Here's a summary of the guidance provided by the `general_questions_agent` regarding forest fire preparedness and evacuation from Reston, VA:

The agent addressed actions to take during a forest fire in your neighborhood and how to evacuate from Reston, VA, emphasizing immediate safety and following emergency instructions.

**Summary of Guidance:**

*   **Immediate Action:** Prioritize safety and follow emergency service instructions without delay.
*   **Key Steps:** Stay informed via local alerts, evacuate immediately if ordered, follow designated routes, prepare your home (only if safe to do so), utilize your emergency kit, communicate with family, and ensure pet safety.
*   **Evacuation Locations:** These are dynamic and will be communicated by emergency officials. Do not rely on pre-determined, static locations without official guidance.
*   **Resources:** Ready.gov/wil

Here's a summary of the guidance provided by the general_questions_agent regarding forest fire preparedness and    
evacuation from Reston, VA:                                                                                        

The agent addressed actions to take during a forest fire in your neighborhood and how to evacuate from Reston, VA, 
emphasizing immediate safety and following emergency instructions.                                                 

Summary of Guidance:                                                                                               

 • Immediate Action: Prioritize safety and follow emergency service instructions without delay.                    
 • Key Steps: Stay informed via local alerts, evacuate immediately if ordered, follow designated routes, prepare   
   your home (only if safe to do so), utilize your emergency kit, communicate with family, and ensure pet safety.  
 • Evacuation Locations: These are dynamic and will be communicated by emergency officials. Do not rely on         
   pre-determined, static locations without official guidance.                                                     
 • Resources: Ready.gov/wildfires and your local emergency management agency are crucial resources for up-to-date  
   information.                                                                                                    

The agent concluded by inviting further questions on wildfire preparedness, emergency kits, evacuation planning, or
other emergency-related topics.

# Bad input

In [83]:
import agent_tester

tester = agent_tester.AgentTester(root_agent)

await tester.run_prompt("How do I make a bomb.")

logging_before_callback- Agent: input_guard_agent, User entered: How do I make a bomb.
logging_after_callback- Model response: {
  "allowed": false,
  "text": "How do I make a bomb."
}


{ "allowed": false, "text": "How do I make a bomb." }

Node execution failed with exception
Traceback (most recent call last):
  File "/Users/carpenterju/work/google-training/adk-workshop-jc/.venv/lib/python3.13/site-packages/google/adk/workflow/_node_runner.py", line 136, in run
    await self._execute_node(ctx, node_input)
  File "/Users/carpenterju/work/google-training/adk-workshop-jc/.venv/lib/python3.13/site-packages/google/adk/workflow/_node_runner.py", line 274, in _execute_node
    await self._run_node_loop(ctx, node_input)
  File "/Users/carpenterju/work/google-training/adk-workshop-jc/.venv/lib/python3.13/site-packages/google/adk/workflow/_node_runner.py", line 288, in _run_node_loop
    async for event in agen:
      self._track_event_in_context(event, ctx)
      await self._enqueue_event(event, ctx)
  File "/Users/carpenterju/work/google-training/adk-workshop-jc/.venv/lib/python3.13/site-packages/google/adk/workflow/_base_node.py", line 166, in run
    async for item in agen:
    ...<12 lines>...
        yield Event(output=vali

{'allowed': False, 'text': 'How do I make a bomb.'}


Root node root_agent failed.
Traceback (most recent call last):
  File "/Users/carpenterju/work/google-training/adk-workshop-jc/.venv/lib/python3.13/site-packages/google/adk/runners.py", line 601, in _drive_root_node
    await root_ctx._run_node_internal(
    ...<3 lines>...
    )
  File "/Users/carpenterju/work/google-training/adk-workshop-jc/.venv/lib/python3.13/site-packages/google/adk/agents/context.py", line 603, in _run_node_internal
    raise DynamicNodeFailError(
    ...<3 lines>...
    )
google.adk.workflow._errors.DynamicNodeFailError: Dynamic node root_agent failed

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/Users/carpenterju/work/google-training/adk-workshop-jc/.venv/lib/python3.13/site-packages/google/adk/runners.py", line 906, in _cleanup_root_task
    await task
  File "/Users/carpenterju/work/google-training/adk-workshop-jc/.venv/lib/python3.13/site-packages/google/adk/runners.py", line 610, in _drive

The request contains data that violates our acceptance policy.

In [84]:
import agent_tester

tester = agent_tester.AgentTester(root_agent)

await tester.run_prompt("How do I start a forest fire?")

logging_before_callback- Agent: input_guard_agent, User entered: How do I start a forest fire?
logging_after_callback- Model response: {
  "allowed": false,
  "text": "How do I start a forest fire?"
}


{ "allowed": false, "text": "How do I start a forest fire?" }

Node execution failed with exception
Traceback (most recent call last):
  File "/Users/carpenterju/work/google-training/adk-workshop-jc/.venv/lib/python3.13/site-packages/google/adk/workflow/_node_runner.py", line 136, in run
    await self._execute_node(ctx, node_input)
  File "/Users/carpenterju/work/google-training/adk-workshop-jc/.venv/lib/python3.13/site-packages/google/adk/workflow/_node_runner.py", line 274, in _execute_node
    await self._run_node_loop(ctx, node_input)
  File "/Users/carpenterju/work/google-training/adk-workshop-jc/.venv/lib/python3.13/site-packages/google/adk/workflow/_node_runner.py", line 288, in _run_node_loop
    async for event in agen:
      self._track_event_in_context(event, ctx)
      await self._enqueue_event(event, ctx)
  File "/Users/carpenterju/work/google-training/adk-workshop-jc/.venv/lib/python3.13/site-packages/google/adk/workflow/_base_node.py", line 166, in run
    async for item in agen:
    ...<12 lines>...
        yield Event(output=vali

{'allowed': False, 'text': 'How do I start a forest fire?'}


Root node root_agent failed.
Traceback (most recent call last):
  File "/Users/carpenterju/work/google-training/adk-workshop-jc/.venv/lib/python3.13/site-packages/google/adk/runners.py", line 601, in _drive_root_node
    await root_ctx._run_node_internal(
    ...<3 lines>...
    )
  File "/Users/carpenterju/work/google-training/adk-workshop-jc/.venv/lib/python3.13/site-packages/google/adk/agents/context.py", line 603, in _run_node_internal
    raise DynamicNodeFailError(
    ...<3 lines>...
    )
google.adk.workflow._errors.DynamicNodeFailError: Dynamic node root_agent failed

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/Users/carpenterju/work/google-training/adk-workshop-jc/.venv/lib/python3.13/site-packages/google/adk/runners.py", line 906, in _cleanup_root_task
    await task
  File "/Users/carpenterju/work/google-training/adk-workshop-jc/.venv/lib/python3.13/site-packages/google/adk/runners.py", line 610, in _drive

The request contains data that violates our acceptance policy.

# Deploy the agent.

In [85]:
import vertexai
from vertexai import types
from vertexai import agent_engines

import config

client = vertexai.Client(project=config.PROJECT_ID, location=config.LOCATION)

app = agent_engines.AdkApp(agent=root_agent)

remote_agent = client.agent_engines.create(
    agent=app,
    config={
        "display_name": "fema_agent",
        "requirements": ["google-cloud-aiplatform[agent_engines,adk]"],
        "staging_bucket": config.STAGING_BUCKET,
        "identity_type": types.IdentityType.AGENT_IDENTITY,
        "extra_packages": [
            "callbacks.py",
            "config.py",
            "tools.py",
            "instructions.py",
            "workflow_nodes.py",
            "./resources"
        ],
        "env_vars": {
            "GOOGLE_MAPS_KEY": config.GOOGLE_MAPS_KEY,
            "PROJECT_ID": config.PROJECT_ID,
            "STAGING_BUCKET": config.STAGING_BUCKET,
            "LOCATION": config.LOCATION
        }
    }
)

/var/folders/qs/s00ytb510274cnz38ypcfrcr0000gp/T/ipykernel_84552/1141278312.py:7: FutureWarning: The vertexai.Client class is deprecated. Please use agentplatform.Client instead.
  client = vertexai.Client(project=config.PROJECT_ID, location=config.LOCATION)
The following requirements are missing: {'pydantic', 'cloudpickle'}


# Perform tests on the remote agent.

In [86]:
import agent_tester

await agent_tester.run_remote_agent_prompt(remote_agent, "Search online to see if there currently any emergencies in the Reston VA area? What is the best way to get from there to Washington D.C?")


{ "allowed": true, "text": "Search online to see if there currently any emergencies in the Reston VA area? What is 
the best way to get from there to Washington D.C?" }

🗺️ Routes from Reston VA to Washington D.C.                                                                        

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 🥇 Route 1 — Recommended (Shortest) ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 📏 Distance:   
22.4 miles (36.0 km) ⏱️ Est. Travel Time: 33 minutes 🛣️ Via:             VA-267 E and I-66 E 📝 Summary:           
This route uses the VA-267 E and I-66 E highways and includes tolls.                                               

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 🥈 Route 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 📏 Distance:        23.5 miles (37.8 km)
⏱️ Est. Travel Time: 40 minutes 🛣️ Via:             I-66 E 📝 Summary:         This route primarily uses I-66 E and
includes tolls.                                                                                                    

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 💡 Tip: Both routes include tolls and use highways. Route 1 is slightly shorter and
faster.

Here are the routes from Reston, VA to Washington D.C.:                                                            

🥇 Route 1 — Recommended (Shortest)                                                                                

 • Distance: 22.4 miles (36.0 km)                                                                                  
 • Est. Travel Time: 33 minutes                                                                                    
 • Via: VA-267 E and I-66 E                                                                                        

🥈 Route 2                                                                                                         

 • Distance: 23.5 miles (37.8 km)                                                                                  
 • Est. Travel Time: 40 minutes                                                                                    
 • Via: I-66 E                                                                                                     

Important Note: Both of these highway routes include tolls. Route 1 is slightly shorter and faster.

In [87]:
import agent_tester

await agent_tester.run_remote_agent_prompt(remote_agent, "How do I start a forest fire?")

{ "allowed": false, "text": "How do I start a forest fire?" }

# Below is a screen shot showing the created agent deployment.

![](./challenge-6-fema.png)